In [ ]:
import numpy as np
import pandas as pd
import os
import joblib
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


DATA_PATH = "Dataset/Train.csv"
SAVE_DIR = "saved_forecasting_model"

N_PAST = 168
EPOCHS = 20
BATCH_SIZE = 64

os.makedirs(SAVE_DIR, exist_ok=True)

Loading saved bottleneck models...



In [ ]:
print("Loading dataset...")

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)


df["TimeStamp"] = pd.to_datetime(
    df["TimeStamp"],
    format="%d/%m/%y %H:%M",
    errors="coerce"
)

df = df.dropna(subset=["TimeStamp"])

df = df.sort_values("TimeStamp").reset_index(drop=True)

Loading dataset...
Dataset Shape: (38373, 10)


In [ ]:
df["Hour"] = df["TimeStamp"].dt.hour
df["Day"] = df["TimeStamp"].dt.day
df["Month"] = df["TimeStamp"].dt.month
df["Year"] = df["TimeStamp"].dt.year
df["Weekday"] = df["TimeStamp"].dt.weekday

df["IsWeekend"] = (
    df["Weekday"] >= 5
).astype(int)


if "Holiday" in df.columns:

    df["IsHoliday"] = (
        df["Holiday"].notnull()
    ).astype(int)

else:

    df["IsHoliday"] = 0

In [ ]:
weather_encoder = LabelEncoder()

if "Weather" in df.columns:

    df["Weather"] = df["Weather"].fillna("Unknown")

    df["Weather_encoded"] = (
        weather_encoder.fit_transform(
            df["Weather"]
        )
    )

else:

    df["Weather_encoded"] = 0


weather_desc_encoder = LabelEncoder()

if "Weather_Desc" in df.columns:

    df["Weather_Desc"] = (
        df["Weather_Desc"].fillna("Unknown")
    )

    df["Weather_Desc_encoded"] = (
        weather_desc_encoder.fit_transform(
            df["Weather_Desc"]
        )
    )

else:

    df["Weather_Desc_encoded"] = 0

In [ ]:
df["Hour_sin"] = np.sin(
    2 * np.pi * df["Hour"] / 24
)

df["Hour_cos"] = np.cos(
    2 * np.pi * df["Hour"] / 24
)


df["Day_sin"] = np.sin(
    2 * np.pi * df["Day"] / 31
)

df["Day_cos"] = np.cos(
    2 * np.pi * df["Day"] / 31
)


df["Month_sin"] = np.sin(
    2 * np.pi * df["Month"] / 12
)

df["Month_cos"] = np.cos(
    2 * np.pi * df["Month"] / 12
)

In [ ]:
df["IsMorning"] = (
    (df["Hour"] >= 6) &
    (df["Hour"] < 12)
).astype(int)


df["IsAfternoon"] = (
    (df["Hour"] >= 12) &
    (df["Hour"] < 18)
).astype(int)


df["IsEvening"] = (
    (df["Hour"] >= 18) &
    (df["Hour"] < 24)
).astype(int)


In [ ]:
columns_to_drop = []

for column in [
    "Date",
    "Holiday",
    "Weather",
    "Weather_Desc"
]:

    if column in df.columns:

        columns_to_drop.append(column)


df = df.drop(
    columns=columns_to_drop
)

df = df.ffill().bfill()

In [ ]:
target_column = "Traffic_Vol"


feature_columns = [

    column

    for column in df.columns

    if column not in [
        "TimeStamp",
        target_column
    ]

]


features = df[
    feature_columns
].copy()


target = df[
    target_column
].copy()


print("\nNumber of Features:", len(feature_columns))

print("\nFeatures Used:")

print(feature_columns)


Number of Features: 22

Features Used:
['Temperature', 'Rainfall_last_hour', 'Snowfall_last_hour', 'Cloud_Cover', 'Hour', 'Day', 'Month', 'Year', 'Weekday', 'IsWeekend', 'IsHoliday', 'Weather_encoded', 'Weather_Desc_encoded', 'Hour_sin', 'Hour_cos', 'Day_sin', 'Day_cos', 'Month_sin', 'Month_cos', 'IsMorning', 'IsAfternoon', 'IsEvening']


In [ ]:
q33 = target.quantile(0.33)

q66 = target.quantile(0.66)


traffic_thresholds = {

    "low_threshold": q33,

    "moderate_threshold": q66

}


feature_scaler = MinMaxScaler()

scaled_features = feature_scaler.fit_transform(
    features
)

In [ ]:
target_scaler = MinMaxScaler()

scaled_target = target_scaler.fit_transform(
    target.values.reshape(-1, 1)
)


In [ ]:
X = []

y = []


for i in range(

    N_PAST,

    len(scaled_features)

):

    X.append(

        scaled_features[
            i - N_PAST:i
        ]

    )

    y.append(

        scaled_target[
            i,
            0
        ]

    )


X = np.array(X)

y = np.array(y)


print("\nX Shape:", X.shape)

print("y Shape:", y.shape)


train_size = int(
    len(X) * 0.90
)


X_train = X[
    :train_size
]

X_test = X[
    train_size:
]


y_train = y[
    :train_size
]

y_test = y[
    train_size:
]



X Shape: (38205, 168, 22)
y Shape: (38205,)


In [ ]:
model = tf.keras.Sequential([

    tf.keras.layers.Input(
        shape=(
            N_PAST,
            X.shape[2]
        )
    ),

    tf.keras.layers.LSTM(
        128,
        return_sequences=True
    ),

    tf.keras.layers.Dropout(
        0.2
    ),

    tf.keras.layers.LSTM(
        64,
        return_sequences=True
    ),

    tf.keras.layers.Dropout(
        0.2
    ),

    tf.keras.layers.LSTM(
        32
    ),

    tf.keras.layers.Dense(
        16,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        1
    )

])

model.compile(

    optimizer="adam",

    loss="mse",

    metrics=[
        "mae"
    ]

)


model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 168, 128)          77312     
                                                                 
 dropout (Dropout)           (None, 168, 128)          0         
                                                                 
 lstm_1 (LSTM)               (None, 168, 64)           49408     
                                                                 
 dropout_1 (Dropout)         (None, 168, 64)           0         
                                                                 
 lstm_2 (LSTM)               (None, 32)                12416     
                                                                 
 dense (Dense)               (None, 16)                528       
                                                                 
 dense_1 (Dense)             (None, 1)                

In [ ]:
history = model.fit(

    X_train,

    y_train,

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    validation_data=(

        X_test,

        y_test

    ),

    verbose=1

)


y_pred_scaled = model.predict(
    X_test
)


y_pred = target_scaler.inverse_transform(
    y_pred_scaled
)


y_actual = target_scaler.inverse_transform(

    y_test.reshape(
        -1,
        1
    )

)


Epoch 1/20


538/538 [==============================] - 140s 247ms/step - loss: 0.0204 - mae: 0.1007 - val_loss: 0.0075 - val_mae: 0.0602
Epoch 2/20
538/538 [==============================] - 117s 218ms/step - loss: 0.0102 - mae: 0.0686 - val_loss: 0.0079 - val_mae: 0.0594
Epoch 3/20
538/538 [==============================] - 117s 218ms/step - loss: 0.0088 - mae: 0.0619 - val_loss: 0.0057 - val_mae: 0.0513
Epoch 4/20
538/538 [==============================] - 118s 219ms/step - loss: 0.0081 - mae: 0.0590 - val_loss: 0.0053 - val_mae: 0.0516
Epoch 5/20
538/538 [==============================] - 559s 1s/step - loss: 0.0074 - mae: 0.0560 - val_loss: 0.0046 - val_mae: 0.0431
Epoch 6/20
538/538 [==============================] - 116s 215ms/step - loss: 0.0068 - mae: 0.0538 - val_loss: 0.0043 - val_mae: 0.0436
Epoch 7/20
538/538 [==============================] - 118s 220ms/step - loss: 0.0064 - mae: 0.0521 - val_loss: 0.0047 - val_mae: 0.0414
Epoch 8/20
538/538 [=============================

In [ ]:
mse = mean_squared_error(

    y_actual,

    y_pred

)


mae = mean_absolute_error(

    y_actual,

    y_pred

)


r2 = r2_score(

    y_actual,

    y_pred

)


print("\n")
print("=" * 60)

print("MODEL PERFORMANCE")

print("=" * 60)

print("MSE:", mse)

print("MAE:", mae)

print("R2 Score:", r2)



MODEL PERFORMANCE
MSE: 267012.0899503144
MAE: 334.85340605403576
R2 Score: 0.9342476398477773


In [ ]:

print("\nSaving model...")


model.save(

    f"{SAVE_DIR}/traffic_lstm.keras"

)

joblib.dump(

    feature_scaler,

    f"{SAVE_DIR}/feature_scaler.pkl"

)


joblib.dump(

    target_scaler,

    f"{SAVE_DIR}/target_scaler.pkl"

)

joblib.dump(

    weather_encoder,

    f"{SAVE_DIR}/weather_encoder.pkl"

)


joblib.dump(

    weather_desc_encoder,

    f"{SAVE_DIR}/weather_desc_encoder.pkl"

)

joblib.dump(

    feature_columns,

    f"{SAVE_DIR}/feature_columns.pkl"

)

joblib.dump(

    N_PAST,

    f"{SAVE_DIR}/n_past.pkl"

)

joblib.dump(

    traffic_thresholds,

    f"{SAVE_DIR}/traffic_thresholds.pkl"

)

df.to_pickle(

    f"{SAVE_DIR}/processed_history.pkl"

)


print("\n")
print("=" * 60)

print("MODEL SAVED SUCCESSFULLY")

print("=" * 60)

print(
    f"Saved Location: {SAVE_DIR}"
)


Saving model...


MODEL SAVED SUCCESSFULLY
Saved Location: saved_forecasting_model
